In [ ]:
# Increase SWAP Memory on Ubuntu
# sudo swapoff -a
# sudo dd if=/dev/zero of=/swapfile bs=1M count=19456
# sudo chmod 600 /swapfile
# sudo mkswap /swapfile
# sudo swapon /swapfile


In [ ]:
# ==============================================================================
# LOGISTIC REGRESSION AND IHT
# ==============================================================================

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================


In [ ]:

import os
import gc
import json
import time
import warnings
from collections import Counter

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.inspection import permutation_importance
from imblearn.under_sampling import InstanceHardnessThreshold

warnings.filterwarnings("default")

# Print messages immediately so long runs show progress in real time.

def log(msg):
    print(msg, flush=True)



In [4]:

# ============================================================================
# CONFIG
# ============================================================================


In [5]:

SEED = 42
RAW_DATA_PATH = os.path.join("..", "data", "processed", "dados_uteis_refreshed_v3_memorysafe.pkl")
OUTPUT_DIR = os.path.join("..", "data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "ausencia"
RUN_TAG = "logreg_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained"

DROP_COLS = [
    "cns_usuario", "ibge_usuario", "cep_usuario", "cep_cnes_exe",
    "ibge_cnes_exe", "lat_cnes", "lon_cnes", "lat_usuario", "lon_usuario",
    "cep_aprox_usuario", "cep_aprox_cnes", "dta_atend", "cod_uf_usuario",
    "cod_uf_cnes", "cidade_usuario", "cidade_cnes"
]

N_OUTER_FOLDS = 10  # z1
N_INNER_FOLDS = 10  # z2
TOP_K_FEATURES = 20

# IHT inside inner/outer training only
# For imbalanced-learn under-sampling, float sampling_strategy means:
# minority_count / majority_count after resampling.
# 0.5 therefore means majority:minority ~= 2:1.
# For a strict 1:1 ratio, use IHT_SAMPLING_STRATEGY = 1.0.
IHT_SAMPLING_STRATEGY = 0.5
IHT_CV = 3

# Memory-safe nested-CV caps. The outer validation fold is never capped.
# These caps only limit the training/calibration side to keep IHT feasible.
OUTER_CALIBRATION_CAP = 800_000
INNER_TRAIN_CAP = 250_000

# search on resampled train, scoring on raw inner validation
SEARCH_SIZE_CAP = 120_000
C_CANDIDATES = np.logspace(-3, 2, 12).tolist()
FIT_INTERCEPT_CANDIDATES = [True, False]

# permutation importance for feature ranking (inner folds only)
PERM_RANK_MAX_SAMPLES = 20000
PERM_RANK_REPEATS = 3

# final performance emphasis
# Youden's J + constraints avoids unstable all-positive/all-negative thresholds.
THRESHOLD_OBJECTIVE = "youden_j"
MANUAL_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
MIN_THRESHOLD = 0.10
MIN_SPECIFICITY_FOR_THRESHOLD = 0.20
MIN_PRECISION_MARGIN_OVER_PREVALENCE = 0.00
MAX_PREDICTED_POSITIVE_RATE = 0.70
MIN_PREDICTED_NEGATIVE_RATE = 0.05
USE_SIGMOID_PROBA_CALIBRATION = True
FINAL_PROBA_CALIBRATION_SIZE = 50_000
FINAL_THRESHOLD_SIZE = 50_000

# optional final importance on outer-best fold / full global model is disabled by default
RUN_OUTER_BEST_FOLD_PERMUTATION = False
OUTER_PERM_MAX_SAMPLES = 50000
OUTER_PERM_REPEATS = 5



In [6]:
# ==============================================================================
# HELPERS
# ==============================================================================

In [ ]:
# Build a dense OneHotEncoder while supporting both old and new sklearn parameter names.

def make_dense_ohe(categories=None):
    kwargs = {"handle_unknown": "ignore"}
    if categories is not None:
        kwargs["categories"] = categories
    try:
        return OneHotEncoder(sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=False, **kwargs)



In [ ]:
# Coerce labels safely to integer 0/1 and fail early if labels are missing or not binary.

def ensure_binary_int_labels(y, name="y"):
    if isinstance(y, pd.Series):
        arr = y.to_numpy()
    else:
        arr = np.asarray(y)

    arr = arr.ravel()

    if pd.isna(arr).any():
        raise ValueError(f"{name} contains missing values.")

    try:
        arr_num = pd.to_numeric(pd.Series(arr), errors="raise").to_numpy()
    except Exception:
        uniques = pd.Series(arr).dropna().unique().tolist()
        if len(uniques) != 2:
            raise ValueError(f"{name} is not binary and could not be coerced safely.")
        mapping = {uniques[0]: 0, uniques[1]: 1}
        arr_num = pd.Series(arr).map(mapping).to_numpy()

    arr_int = arr_num.astype(np.int64)
    unique_vals = np.unique(arr_int)
    valid_sets = [np.array([0, 1]), np.array([0]), np.array([1])]
    if not any(np.array_equal(unique_vals, s) for s in valid_sets):
        raise ValueError(f"{name} must contain only 0/1 after coercion. Found: {unique_vals}")

    return arr_int


In [ ]:
# Return TN, FP, FN, TP with fixed 0/1 labels, even when one class is absent.

def safe_binary_confusion(y_true, y_pred):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape != (2, 2):
        padded = np.zeros((2, 2), dtype=int)
        padded[:cm.shape[0], :cm.shape[1]] = cm
        cm = padded
    tn, fp, fn, tp = cm.ravel()
    return tn, fp, fn, tp


In [ ]:
# Compute specificity, TN / (TN + FP), with a safe zero-denominator fallback.

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    denom = tn + fp
    return 0.0 if denom == 0 else tn / denom



In [ ]:
# Compute negative predictive value and return NaN when no predicted negatives exist.

def npv_score(y_true, y_pred):
    """Negative Predictive Value: TN / (TN + FN).

    If there are no predicted negatives, NPV is undefined. Returning np.nan is
    more honest than returning 0.0, because 0.0 would mean all predicted
    negatives were wrong.
    """
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    denom = tn + fn
    return np.nan if denom == 0 else tn / denom



In [ ]:
# Compute ROC AUC, returning NaN instead of crashing when AUC is undefined.

def safe_roc_auc(y_true, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    try:
        return roc_auc_score(y_true, y_proba)
    except ValueError:
        return np.nan


In [ ]:
# Collect fold metrics, confusion counts, class rates, and degenerate-prediction flags.

def calculate_metrics(y_true, y_pred, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

    real_neg = tn + fp
    real_pos = tp + fn
    pred_neg = tn + fn
    pred_pos = tp + fp
    total = real_neg + real_pos

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": safe_roc_auc(y_true, y_proba),

        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),

        "Real_Positive": int(real_pos),
        "Real_Negative": int(real_neg),
        "Predicted_Positive": int(pred_pos),
        "Predicted_Negative": int(pred_neg),
        "Real_Positive_Rate": real_pos / total if total else np.nan,
        "Real_Negative_Rate": real_neg / total if total else np.nan,
        "Predicted_Positive_Rate": pred_pos / total if total else np.nan,
        "Predicted_Negative_Rate": pred_neg / total if total else np.nan,
        "Has_No_Predicted_Negatives": bool(pred_neg == 0),
        "Has_No_Predicted_Positives": bool(pred_pos == 0),
    }



In [ ]:
# Log total samples, class counts, percentages, and imbalance ratio.

def print_class_distribution(name, y):
    y = ensure_binary_int_labels(y, name)
    total = len(y)

    unique, counts = np.unique(y, return_counts=True)
    count_map = {int(k): int(v) for k, v in zip(unique, counts)}

    c0 = count_map.get(0, 0)
    c1 = count_map.get(1, 0)

    p0 = (c0 / total * 100) if total else 0.0
    p1 = (c1 / total * 100) if total else 0.0

    ratio = np.inf if min(c0, c1) == 0 else max(c0, c1) / min(c0, c1)

    log(f"\n{name}")
    log("-" * len(name))
    log(f"Total: {total:,}".replace(",", "."))
    log(f"Class 0: {c0:,} ({p0:.4f}%)".replace(",", "."))
    log(f"Class 1: {c1:,} ({p1:.4f}%)".replace(",", "."))
    if np.isinf(ratio):
        log("Imbalance ratio: infinito")
    else:
        log(f"Imbalance ratio (maj/min): {ratio:.4f}:1")



In [ ]:
# Check that manually supplied thresholds are inside (0, 1) and remove duplicates.

def validate_manual_thresholds(thresholds):
    valid = []
    for t in thresholds:
        t = float(t)
        if not (0.0 < t < 1.0):
            raise ValueError(f"Threshold {t} is invalid. Thresholds must be between 0 and 1.")
        valid.append(t)
    return sorted(set(valid))



In [ ]:
# Select the best probability threshold while blocking degenerate all-positive or no-negative solutions.

def optimize_threshold_constrained(
    y_true,
    y_proba,
    objective="youden_j",
    min_threshold=MIN_THRESHOLD,
    min_specificity=MIN_SPECIFICITY_FOR_THRESHOLD,
    min_precision=None,
    precision_margin_over_prevalence=MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    max_predicted_positive_rate=MAX_PREDICTED_POSITIVE_RATE,
    min_predicted_negative_rate=MIN_PREDICTED_NEGATIVE_RATE,
):
    """Choose a threshold while blocking pathological decision rules."""
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_proba = np.asarray(y_proba, dtype=np.float64)

    prevalence = float(np.mean(y_true))
    if min_precision is None:
        min_precision = prevalence + precision_margin_over_prevalence

    grid = np.linspace(min_threshold, 0.99, 90)
    quantiles = np.quantile(y_proba, np.linspace(0.01, 0.99, 99))
    manual = validate_manual_thresholds(MANUAL_THRESHOLDS)
    thresholds = np.unique(np.concatenate([grid, quantiles, manual]))
    thresholds = thresholds[(thresholds >= min_threshold) & (thresholds < 1.0)]

    rows = []
    for thr in thresholds:
        y_pred = (y_proba >= thr).astype(np.uint8)
        tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

        total = len(y_true)
        pred_pos = tp + fp
        pred_neg = tn + fn
        pred_pos_rate = pred_pos / total if total else np.nan
        pred_neg_rate = pred_neg / total if total else np.nan

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        specificity = specificity_score(y_true, y_pred)
        npv = npv_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        youden_j = recall + specificity - 1.0
        balanced_accuracy = (recall + specificity) / 2.0

        passes_constraints = (
            specificity >= min_specificity
            and precision >= min_precision
            and pred_pos_rate <= max_predicted_positive_rate
            and pred_neg_rate >= min_predicted_negative_rate
        )

        rows.append({
            "Threshold": float(thr),
            "Precision": float(precision),
            "Recall": float(recall),
            "Specificity": float(specificity),
            "NPV": float(npv) if not pd.isna(npv) else np.nan,
            "F1": float(f1),
            "Youden_J": float(youden_j),
            "Balanced_Accuracy": float(balanced_accuracy),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
            "Predicted_Positive": int(pred_pos),
            "Predicted_Negative": int(pred_neg),
            "Predicted_Positive_Rate": float(pred_pos_rate),
            "Predicted_Negative_Rate": float(pred_neg_rate),
            "Min_Precision_Required": float(min_precision),
            "Min_Specificity_Required": float(min_specificity),
            "Max_Predicted_Positive_Rate_Allowed": float(max_predicted_positive_rate),
            "Min_Predicted_Negative_Rate_Required": float(min_predicted_negative_rate),
            "Passes_Constraints": bool(passes_constraints),
        })

    df_thr = pd.DataFrame(rows)
    if objective.lower() == "f1":
        score_col = "F1"
    elif objective.lower() == "youden_j":
        score_col = "Youden_J"
    else:
        raise ValueError("objective must be 'f1' or 'youden_j'")

    df_valid = df_thr[df_thr["Passes_Constraints"]].copy()
    threshold_selection_mode = "constrained"

    if len(df_valid) == 0:
        df_valid = df_thr[(df_thr["Predicted_Negative"] > 0) & (df_thr["Specificity"] > 0)].copy()
        threshold_selection_mode = "fallback_non_degenerate"

    if len(df_valid) == 0:
        df_valid = df_thr.copy()
        threshold_selection_mode = "fallback_all_thresholds"

    best_row = (
        df_valid.sort_values(
            by=[score_col, "Specificity", "Precision", "Recall", "Threshold"],
            ascending=[False, False, False, False, True],
        )
        .iloc[0]
        .to_dict()
    )
    best_row["Threshold_Selection_Mode"] = threshold_selection_mode
    best_row["Threshold_Valid_Candidates"] = int(len(df_valid))
    best_row["Threshold_Total_Candidates"] = int(len(df_thr))
    return best_row, df_thr


In [ ]:
# Clip probabilities away from 0 and 1 so logit calibration stays numerically stable.

def clip_proba(p, eps=1e-6):
    return np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)



In [ ]:
# Convert probabilities to log-odds for sigmoid/Platt-style calibration.

def logit(p):
    p = clip_proba(p)
    return np.log(p / (1.0 - p))



In [ ]:
# Fit a one-dimensional logistic calibrator on held-out natural-distribution probabilities.

def fit_sigmoid_calibrator(y_calib, proba_calib, seed=SEED):
    y_calib = ensure_binary_int_labels(y_calib, "y_calib")
    if len(np.unique(y_calib)) < 2:
        return None
    z = logit(proba_calib).reshape(-1, 1)
    calibrator = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=seed)
    calibrator.fit(z, y_calib)
    return calibrator



In [ ]:
# Apply the fitted probability calibrator, or return raw probabilities if calibration was skipped.

def apply_sigmoid_calibrator(calibrator, proba):
    if calibrator is None:
        return np.asarray(proba, dtype=np.float64)
    z = logit(proba).reshape(-1, 1)
    return calibrator.predict_proba(z)[:, 1]


In [ ]:
# Create a stratified capped subset to keep large operations memory-safe.

def make_cap_subset_indices(y, cap, seed):
    y = ensure_binary_int_labels(y, "y_cap")
    if cap is None or len(y) <= cap:
        return np.arange(len(y))
    indices = np.arange(len(y))
    _, idx_small = train_test_split(
        indices,
        test_size=cap,
        stratify=y,
        random_state=seed,
    )
    return idx_small


In [ ]:
# Split outer-training data into final-fit, calibration, and threshold sets without touching outer validation.

def split_final_train_calib_threshold_indices(
    y,
    calib_size=FINAL_PROBA_CALIBRATION_SIZE,
    threshold_size=FINAL_THRESHOLD_SIZE,
    seed=SEED,
):
    """Split the outer calibration side into train/calibration/threshold partitions.

    The outer validation fold is never used for fitting the calibrator or selecting
    the threshold.
    """
    y = ensure_binary_int_labels(y, "y_final_split")
    idx_all = np.arange(len(y))

    requested_holdout = int(calib_size or 0) + int(threshold_size or 0)
    if requested_holdout <= 0 or len(y) < 10:
        return idx_all, np.array([], dtype=int), np.array([], dtype=int)

    max_holdout = max(2, len(y) // 2)
    total_holdout = min(requested_holdout, max_holdout)

    idx_train, idx_temp = train_test_split(
        idx_all,
        test_size=total_holdout,
        stratify=y,
        random_state=seed,
    )

    if len(idx_temp) < 4:
        return idx_train, idx_temp, idx_temp

    threshold_n = min(int(threshold_size or 0), len(idx_temp) - 2)
    if threshold_n <= 0:
        return idx_train, idx_temp, idx_temp

    idx_calib, idx_threshold = train_test_split(
        idx_temp,
        test_size=threshold_n,
        stratify=y[idx_temp],
        random_state=seed + 1,
    )
    return idx_train, idx_calib, idx_threshold



In [ ]:
# Build a stratified smaller training subset for faster hyperparameter search.

def make_search_subset(X_train, y_train, max_size, seed):
    y_train = ensure_binary_int_labels(y_train, "y_search")
    if len(X_train) <= max_size:
        return X_train, y_train
    indices = np.arange(len(X_train))
    _, idx_small = train_test_split(
        indices,
        test_size=max_size,
        stratify=y_train,
        random_state=seed,
    )
    return X_train[idx_small], y_train[idx_small]



In [ ]:
# Build a stratified smaller validation subset for faster permutation importance.

def make_perm_subset(X, y, max_size, seed):
    y = ensure_binary_int_labels(y, "y_perm_subset")
    if len(X) <= max_size:
        return X, y
    indices = np.arange(len(X))
    _, idx_small = train_test_split(
        indices,
        test_size=max_size,
        stratify=y,
        random_state=seed,
    )
    return X[idx_small], y[idx_small]



In [ ]:
# Generate the logistic-regression grid over C and fit_intercept values.

def build_candidate_list():
    all_candidates = []
    for c in C_CANDIDATES:
        for fit_intercept in FIT_INTERCEPT_CANDIDATES:
            all_candidates.append({
                "C": float(c),
                "fit_intercept": bool(fit_intercept),
                "Candidate_Key": f"C={float(c):.12g}|fit_intercept={bool(fit_intercept)}",
            })
    return all_candidates



In [ ]:
# Scorer used by permutation importance, based on predicted class-1 probabilities.

def roc_auc_proba_scorer(estimator, X, y):
    y = ensure_binary_int_labels(y, "y_score")
    y_proba = estimator.predict_proba(X)[:, 1]
    return roc_auc_score(y, y_proba)


In [ ]:
# Learn numeric columns, categorical columns, and category levels from the outer training fold only.

def build_outer_schema(X_outer_raw):
    num_feats = X_outer_raw.select_dtypes(include=["number"]).columns.tolist()
    cat_feats = X_outer_raw.select_dtypes(include=["category", "object", "string", "bool"]).columns.tolist()
    cat_categories = [pd.Index(X_outer_raw[col].dropna().unique()).tolist() for col in cat_feats]
    return num_feats, cat_feats, cat_categories



In [ ]:
# Build MinMax scaling plus fixed-category one-hot preprocessing to avoid leakage.

def build_preprocessor_from_schema(num_feats, cat_feats, cat_categories):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), num_feats),
            ("cat", make_dense_ohe(categories=cat_categories), cat_feats),
        ],
        verbose_feature_names_out=False,
    )
    return preprocessor



In [ ]:
# Apply Instance Hardness Threshold undersampling only on training data and log the before/after balance.

def apply_iht(X_train, y_train, seed):
    y_train = ensure_binary_int_labels(y_train, "y_train_for_iht")
    before_c0 = int(np.sum(y_train == 0))
    before_c1 = int(np.sum(y_train == 1))

    hardness_estimator = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        max_iter=1000,
        tol=1e-3,
        random_state=seed,
        n_jobs=1,
    )
    iht = InstanceHardnessThreshold(
        estimator=hardness_estimator,
        sampling_strategy=IHT_SAMPLING_STRATEGY,
        cv=IHT_CV,
        random_state=seed,
        n_jobs=1,
    )
    start = time.time()
    X_res, y_res = iht.fit_resample(X_train, y_train)
    elapsed_min = (time.time() - start) / 60.0
    X_res = X_res.astype(np.float32)
    y_res = ensure_binary_int_labels(y_res, "y_resampled_iht")

    after_c0 = int(np.sum(y_res == 0))
    after_c1 = int(np.sum(y_res == 1))
    after_ratio = np.inf if min(after_c0, after_c1) == 0 else max(after_c0, after_c1) / min(after_c0, after_c1)
    log(
        "IHT distribution | "
        f"before: class0={before_c0:,}, class1={before_c1:,} | "
        f"after: class0={after_c0:,}, class1={after_c1:,} | "
        f"maj/min={after_ratio:.4f}:1"
    )
    return X_res, y_res, iht, elapsed_min



In [ ]:
# Fit each logistic candidate on the search set and choose the one with best validation AUC.

def evaluate_candidate_grid_on_validation(X_search, y_search, X_val, y_val, seed):
    candidate_list = build_candidate_list()
    rows = []
    best_score = -np.inf
    best_params = None

    for params in candidate_list:
        clf = LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            max_iter=2000,
            random_state=seed,
            C=params["C"],
            fit_intercept=params["fit_intercept"],
        )
        clf.fit(X_search, y_search)
        y_val_proba = clf.predict_proba(X_val)[:, 1]
        val_auc = safe_roc_auc(y_val, y_val_proba)
        rows.append({
            "Candidate_Key": params["Candidate_Key"],
            "C": params["C"],
            "fit_intercept": params["fit_intercept"],
            "Validation_AUC": val_auc,
        })
        if np.isfinite(val_auc) and val_auc > best_score:
            best_score = val_auc
            best_params = {"C": params["C"], "fit_intercept": params["fit_intercept"]}

    if best_params is None:
        raise RuntimeError("No valid hyperparameter candidate produced a valid AUC score.")

    return best_params, best_score, pd.DataFrame(rows)



In [ ]:
# Create the final L2 logistic regression model with the selected C and intercept option.

def fit_logreg(C, fit_intercept, seed):
    return LogisticRegression(
        solver="lbfgs",
        penalty="l2",
        max_iter=2000,
        random_state=seed,
        C=C,
        fit_intercept=fit_intercept,
    )


In [ ]:
# Compute AUC-based permutation importance and return both aligned vector and ranked table.

def compute_permutation_importance_vector(model, X_val, y_val, feature_names, seed):
    X_perm, y_perm = make_perm_subset(X_val, y_val, PERM_RANK_MAX_SAMPLES, seed)
    perm_results = permutation_importance(
        estimator=model,
        X=X_perm,
        y=y_perm,
        scoring=roc_auc_proba_scorer,
        n_repeats=PERM_RANK_REPEATS,
        random_state=seed,
        n_jobs=1,
        max_samples=1.0,
    )
    df_perm = pd.DataFrame({
        "Feature": feature_names,
        "Importance_Mean": perm_results.importances_mean,
        "Importance_Std": perm_results.importances_std,
    }).sort_values(by="Importance_Mean", ascending=False).reset_index(drop=True)
    importance_vector = df_perm.set_index("Feature")["Importance_Mean"].reindex(feature_names).fillna(0.0).to_numpy()
    return importance_vector, df_perm


In [ ]:
# Summarize outer-fold metrics with mean, standard deviation, min, max, and missing counts.

def summarize_outer_results(df_outer):
    metric_cols = [
        "Accuracy", "Precision", "Recall", "Specificity", "NPV", "F1", "ROC_AUC",
        "Threshold", "Selected_Threshold", "Inner_Median_Threshold",
        "Predicted_Positive_Rate", "Predicted_Negative_Rate",
        "Inner_Best_C", "Inner_IHT_Time_Min_Mean", "Outer_IHT_Time_Min",
    ]
    rows = []
    for col in metric_cols:
        if col in df_outer.columns:
            rows.append({
                "Metric": col,
                "Mean": df_outer[col].mean(skipna=True),
                "Std": df_outer[col].std(ddof=1, skipna=True),
                "Min": df_outer[col].min(skipna=True),
                "Max": df_outer[col].max(skipna=True),
                "Valid_Folds": int(df_outer[col].notna().sum()),
                "Missing_or_Undefined_Folds": int(df_outer[col].isna().sum()),
            })
    return pd.DataFrame(rows)



In [ ]:
# Pool TP, TN, FP, and FN across folds and recompute global confusion-derived metrics.

def summarize_pooled_confusion(df_outer):
    required = ["TP", "TN", "FP", "FN"]
    if not all(c in df_outer.columns for c in required):
        return pd.DataFrame()

    total_tp = int(df_outer["TP"].sum())
    total_tn = int(df_outer["TN"].sum())
    total_fp = int(df_outer["FP"].sum())
    total_fn = int(df_outer["FN"].sum())
    total = total_tp + total_tn + total_fp + total_fn

    accuracy = (total_tp + total_tn) / total if total else np.nan
    precision = np.nan if (total_tp + total_fp) == 0 else total_tp / (total_tp + total_fp)
    recall = np.nan if (total_tp + total_fn) == 0 else total_tp / (total_tp + total_fn)
    specificity = np.nan if (total_tn + total_fp) == 0 else total_tn / (total_tn + total_fp)
    npv = np.nan if (total_tn + total_fn) == 0 else total_tn / (total_tn + total_fn)
    f1 = np.nan if pd.isna(precision) or pd.isna(recall) or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    balanced_accuracy = np.nan if pd.isna(recall) or pd.isna(specificity) else (recall + specificity) / 2.0

    return pd.DataFrame([{
        "TP": total_tp,
        "TN": total_tn,
        "FP": total_fp,
        "FN": total_fn,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "NPV": npv,
        "F1": f1,
        "Balanced_Accuracy": balanced_accuracy,
        "Predicted_Positive": total_tp + total_fp,
        "Predicted_Negative": total_tn + total_fn,
        "Real_Positive": total_tp + total_fn,
        "Real_Negative": total_tn + total_fp,
        "Predicted_Positive_Rate": (total_tp + total_fp) / total if total else np.nan,
        "Predicted_Negative_Rate": (total_tn + total_fn) / total if total else np.nan,
    }])



In [35]:

# ============================================================================
# MAIN
# ============================================================================


In [36]:

log(
    f"--- CONFIGURATION ---\n"
    f"RUN_TAG: {RUN_TAG}\n"
    f"RAW_DATA_PATH: {RAW_DATA_PATH}\n"
    f"TARGET_COL: {TARGET_COL}\n"
    f"N_OUTER_FOLDS: {N_OUTER_FOLDS}\n"
    f"N_INNER_FOLDS: {N_INNER_FOLDS}\n"
    f"TOP_K_FEATURES: {TOP_K_FEATURES}\n"
    f"IHT_SAMPLING_STRATEGY: {IHT_SAMPLING_STRATEGY}\n"
    f"IHT_CV: {IHT_CV}\n"
    f"OUTER_CALIBRATION_CAP: {OUTER_CALIBRATION_CAP}\n"
    f"INNER_TRAIN_CAP: {INNER_TRAIN_CAP}\n"
    f"NUMERIC_SCALER: MinMaxScaler\n"
    f"SEARCH_SIZE_CAP: {SEARCH_SIZE_CAP}\n"
    f"PERM_RANK_MAX_SAMPLES: {PERM_RANK_MAX_SAMPLES}\n"
    f"PERM_RANK_REPEATS: {PERM_RANK_REPEATS}\n"
    f"THRESHOLD_OBJECTIVE: {THRESHOLD_OBJECTIVE}\n"
    f"MIN_THRESHOLD: {MIN_THRESHOLD}\n"
    f"MIN_SPECIFICITY_FOR_THRESHOLD: {MIN_SPECIFICITY_FOR_THRESHOLD}\n"
    f"MAX_PREDICTED_POSITIVE_RATE: {MAX_PREDICTED_POSITIVE_RATE}\n"
    f"MIN_PREDICTED_NEGATIVE_RATE: {MIN_PREDICTED_NEGATIVE_RATE}\n"
    f"USE_SIGMOID_PROBA_CALIBRATION: {USE_SIGMOID_PROBA_CALIBRATION}\n"
    f"FINAL_PROBA_CALIBRATION_SIZE: {FINAL_PROBA_CALIBRATION_SIZE}\n"
    f"FINAL_THRESHOLD_SIZE: {FINAL_THRESHOLD_SIZE}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n"
    f"---------------------"
)


--- CONFIGURATION ---
RUN_TAG: logreg_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained
RAW_DATA_PATH: ../data/processed/dados_uteis_refreshed_v3_memorysafe.pkl
TARGET_COL: ausencia
N_OUTER_FOLDS: 10
N_INNER_FOLDS: 10
TOP_K_FEATURES: 20
IHT_SAMPLING_STRATEGY: 0.5
IHT_CV: 3
OUTER_CALIBRATION_CAP: 800000
INNER_TRAIN_CAP: 250000
NUMERIC_SCALER: MinMaxScaler
SEARCH_SIZE_CAP: 120000
PERM_RANK_MAX_SAMPLES: 20000
PERM_RANK_REPEATS: 3
THRESHOLD_OBJECTIVE: youden_j
MIN_THRESHOLD: 0.1
MIN_SPECIFICITY_FOR_THRESHOLD: 0.2
MAX_PREDICTED_POSITIVE_RATE: 0.7
MIN_PREDICTED_NEGATIVE_RATE: 0.05
USE_SIGMOID_PROBA_CALIBRATION: True
FINAL_PROBA_CALIBRATION_SIZE: 50000
FINAL_THRESHOLD_SIZE: 50000
OUTPUT_DIR: ../data/processed
---------------------


In [37]:

log("Loading raw dataset...")
df = pd.read_pickle(RAW_DATA_PATH)
if not isinstance(df, pd.DataFrame):
    raise TypeError(f"Loaded object is {type(df).__name__}, not DataFrame")
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found")

log(f"Raw DataFrame shape: {df.shape}")
y_full = df[TARGET_COL].copy()
feature_drop_cols = [c for c in DROP_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in feature_drop_cols + [TARGET_COL]]
log(f"Number of feature columns before preprocessing: {len(feature_cols)}")
print_class_distribution("FULL DATASET DISTRIBUTION", y_full)


Loading raw dataset...
Raw DataFrame shape: (7113798, 42)
Number of feature columns before preprocessing: 25

FULL DATASET DISTRIBUTION
-------------------------
Total: 7.113.798
Class 0: 6.069.245 (85.3165%)
Class 1: 1.044.553 (14.6835%)
Imbalance ratio (maj/min): 5.8104:1


In [38]:


outer_skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED)

outer_results = []
outer_feature_rows = []
outer_inner_candidate_rows = []
outer_inner_feature_rows = []
outer_threshold_rows = []
confusion_rows = []
global_outer_selected_counter = Counter()

best_outer_fold_auc = -np.inf
best_outer_artifacts = None

for outer_fold, (cal_idx_full, val_idx) in enumerate(
    outer_skf.split(np.zeros(len(y_full)), ensure_binary_int_labels(y_full)),
    start=1,
):
    outer_start = time.time()
    log("\n" + "#" * 90)
    log(f"STARTING OUTER FOLD {outer_fold}/{N_OUTER_FOLDS}")
    log("#" * 90)

    y_cal_full = ensure_binary_int_labels(df.iloc[cal_idx_full][TARGET_COL], f"y_cal_full_outer_{outer_fold}")
    idx_cap_outer = make_cap_subset_indices(y_cal_full, OUTER_CALIBRATION_CAP, SEED + outer_fold)
    cal_idx = cal_idx_full[idx_cap_outer]

    X_cal_raw = df.iloc[cal_idx][feature_cols].copy()
    y_cal = ensure_binary_int_labels(df.iloc[cal_idx][TARGET_COL], f"y_cal_outer_{outer_fold}")
    X_val_raw = df.iloc[val_idx][feature_cols].copy()
    y_val = ensure_binary_int_labels(df.iloc[val_idx][TARGET_COL], f"y_val_outer_{outer_fold}")

    print_class_distribution(f"OUTER FOLD {outer_fold} - CALIBRATION", y_cal)
    print_class_distribution(f"OUTER FOLD {outer_fold} - VALIDATION", y_val)

    # fixed schema inside outer fold so inner-fold feature names align
    num_feats, cat_feats, cat_categories = build_outer_schema(X_cal_raw)
    log(f"Outer fold {outer_fold} schema | numeric={len(num_feats)} | categorical={len(cat_feats)}")

    inner_skf = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED + outer_fold)

    inner_importance_vectors = []
    inner_thresholds = []
    inner_iht_times = []
    selected_counter_inner = Counter()
    candidate_auc_accumulator = {}

    for inner_fold, (in_train_idx_full, in_test_idx) in enumerate(
        inner_skf.split(np.zeros(len(y_cal)), y_cal),
        start=1,
    ):
        log("\n" + "=" * 80)
        log(f"OUTER {outer_fold} | INNER {inner_fold}/{N_INNER_FOLDS}")
        log("=" * 80)

        y_in_train_full = ensure_binary_int_labels(y_cal[in_train_idx_full], f"y_in_train_full_{outer_fold}_{inner_fold}")
        idx_cap_inner = make_cap_subset_indices(
            y_in_train_full,
            INNER_TRAIN_CAP,
            SEED + outer_fold * 100 + inner_fold,
        )
        in_train_idx = in_train_idx_full[idx_cap_inner]

        X_in_train_raw = X_cal_raw.iloc[in_train_idx].copy()
        y_in_train = ensure_binary_int_labels(y_cal[in_train_idx], f"y_in_train_{outer_fold}_{inner_fold}")
        X_in_test_raw = X_cal_raw.iloc[in_test_idx].copy()
        y_in_test = ensure_binary_int_labels(y_cal[in_test_idx], f"y_in_test_{outer_fold}_{inner_fold}")

        preprocessor_inner = build_preprocessor_from_schema(num_feats, cat_feats, cat_categories)
        preprocessor_inner.fit(X_in_train_raw)
        feature_names_inner = preprocessor_inner.get_feature_names_out().tolist()

        X_in_train = preprocessor_inner.transform(X_in_train_raw).astype(np.float32)
        X_in_test = preprocessor_inner.transform(X_in_test_raw).astype(np.float32)

        X_in_train_iht, y_in_train_iht, _, iht_time_min = apply_iht(X_in_train, y_in_train, SEED + outer_fold * 100 + inner_fold)
        inner_iht_times.append(iht_time_min)
        log(f"Inner fold IHT time: {iht_time_min:.2f}m")

        X_search, y_search = make_search_subset(
            X_in_train_iht,
            y_in_train_iht,
            SEARCH_SIZE_CAP,
            SEED + outer_fold * 1000 + inner_fold,
        )

        best_params_inner, best_auc_inner, df_search_inner = evaluate_candidate_grid_on_validation(
            X_search,
            y_search,
            X_in_test,
            y_in_test,
            SEED + outer_fold * 10000 + inner_fold,
        )
        df_search_inner["Outer_Fold"] = outer_fold
        df_search_inner["Inner_Fold"] = inner_fold
        outer_inner_candidate_rows.append(df_search_inner)

        # accumulate candidate performance for consolidation of best params
        for _, row in df_search_inner.iterrows():
            key = row["Candidate_Key"]
            if key not in candidate_auc_accumulator:
                candidate_auc_accumulator[key] = {
                    "C": row["C"],
                    "fit_intercept": row["fit_intercept"],
                    "AUCs": [],
                }
            candidate_auc_accumulator[key]["AUCs"].append(row["Validation_AUC"])

        inner_model = fit_logreg(
            C=best_params_inner["C"],
            fit_intercept=best_params_inner["fit_intercept"],
            seed=SEED + outer_fold * 100000 + inner_fold,
        )
        inner_model.fit(X_in_train_iht, y_in_train_iht)

        y_in_test_proba = inner_model.predict_proba(X_in_test)[:, 1]
        best_thr_inner, _ = optimize_threshold_constrained(y_in_test, y_in_test_proba, THRESHOLD_OBJECTIVE)
        inner_thresholds.append(float(best_thr_inner["Threshold"]))

        importance_vector, df_perm_inner = compute_permutation_importance_vector(
            model=inner_model,
            X_val=X_in_test,
            y_val=y_in_test,
            feature_names=feature_names_inner,
            seed=SEED + outer_fold * 1000000 + inner_fold,
        )
        inner_importance_vectors.append(importance_vector)

        top_idx_inner = np.argsort(-importance_vector)[:TOP_K_FEATURES]
        top_names_inner = [feature_names_inner[i] for i in top_idx_inner]
        selected_counter_inner.update(top_names_inner)

        for rank_pos, idx_feat in enumerate(top_idx_inner, start=1):
            outer_inner_feature_rows.append({
                "Outer_Fold": outer_fold,
                "Inner_Fold": inner_fold,
                "Rank_Within_Inner": rank_pos,
                "Feature_Index": int(idx_feat),
                "Feature": feature_names_inner[idx_feat],
                "Importance_Mean": float(importance_vector[idx_feat]),
                "Best_Params_C": best_params_inner["C"],
                "Best_Params_fit_intercept": best_params_inner["fit_intercept"],
                "Best_Inner_AUC": best_auc_inner,
                "Best_Inner_Threshold": float(best_thr_inner["Threshold"]),
            })

        del X_in_train_raw, X_in_test_raw, X_in_train, X_in_test, X_in_train_iht, y_in_train_iht, X_search, y_search, inner_model
        gc.collect()

    # ------------------ Consolidation inside outer fold ------------------
    importance_matrix = np.vstack(inner_importance_vectors)
    mean_importance = importance_matrix.mean(axis=0)
    std_importance = importance_matrix.std(axis=0, ddof=1) if importance_matrix.shape[0] > 1 else np.zeros_like(mean_importance)
    feature_names_outer = feature_names_inner

    inner_feature_summary = pd.DataFrame({
        "Feature_Index": np.arange(len(feature_names_outer)),
        "Feature": feature_names_outer,
        "Mean_Inner_Importance": mean_importance,
        "Std_Inner_Importance": std_importance,
        "Selection_Frequency_Inner": [selected_counter_inner.get(f, 0) for f in feature_names_outer],
    }).sort_values(
        by=["Mean_Inner_Importance", "Selection_Frequency_Inner"],
        ascending=[False, False],
    ).reset_index(drop=True)

    consolidated_idx = inner_feature_summary.head(TOP_K_FEATURES)["Feature_Index"].to_numpy(dtype=int)
    consolidated_features = inner_feature_summary.head(TOP_K_FEATURES)["Feature"].tolist()

    # consolidate best params by mean inner AUC over the same candidate grid
    candidate_summary_rows = []
    for key, d in candidate_auc_accumulator.items():
        aucs = np.array(d["AUCs"], dtype=float)
        candidate_summary_rows.append({
            "Outer_Fold": outer_fold,
            "Candidate_Key": key,
            "C": d["C"],
            "fit_intercept": d["fit_intercept"],
            "Mean_Inner_AUC": float(np.nanmean(aucs)),
            "Std_Inner_AUC": float(np.nanstd(aucs, ddof=1)) if len(aucs) > 1 else 0.0,
        })
    df_candidate_summary_outer = pd.DataFrame(candidate_summary_rows).sort_values(
        by=["Mean_Inner_AUC", "Std_Inner_AUC"],
        ascending=[False, True],
    ).reset_index(drop=True)
    best_params_outer = {
        "C": float(df_candidate_summary_outer.iloc[0]["C"]),
        "fit_intercept": bool(df_candidate_summary_outer.iloc[0]["fit_intercept"]),
    }

    outer_inner_candidate_rows.append(df_candidate_summary_outer.assign(Inner_Fold="summary"))

    consolidated_threshold = float(np.median(inner_thresholds))
    log(f"Outer fold {outer_fold} consolidated threshold (median inner best): {consolidated_threshold:.6f}")
    log(f"Outer fold {outer_fold} consolidated params: {best_params_outer}")
    log(f"Outer fold {outer_fold} consolidated top-{TOP_K_FEATURES} features: {consolidated_features}")

    for rank_pos, row in inner_feature_summary.head(TOP_K_FEATURES).reset_index(drop=True).iterrows():
        outer_feature_rows.append({
            "Outer_Fold": outer_fold,
            "Rank_Within_Outer": rank_pos + 1,
            "Feature_Index": int(row["Feature_Index"]),
            "Feature": row["Feature"],
            "Mean_Inner_Importance": float(row["Mean_Inner_Importance"]),
            "Std_Inner_Importance": float(row["Std_Inner_Importance"]),
            "Selection_Frequency_Inner": int(row["Selection_Frequency_Inner"]),
        })
        global_outer_selected_counter[row["Feature"]] += 1

    # ------------------ Final outer-fold training / calibration / threshold tuning ------------------
    # The outer validation fold remains untouched until final evaluation.
    idx_final_train, idx_proba_calib, idx_threshold = split_final_train_calib_threshold_indices(
        y_cal,
        calib_size=FINAL_PROBA_CALIBRATION_SIZE,
        threshold_size=FINAL_THRESHOLD_SIZE,
        seed=SEED + outer_fold * 5000,
    )

    X_final_train_raw = X_cal_raw.iloc[idx_final_train].copy()
    y_final_train = y_cal[idx_final_train]
    X_proba_calib_raw = X_cal_raw.iloc[idx_proba_calib].copy()
    y_proba_calib = y_cal[idx_proba_calib]
    X_threshold_raw = X_cal_raw.iloc[idx_threshold].copy()
    y_threshold = y_cal[idx_threshold]

    log(f"Outer fold {outer_fold} final train size: {len(y_final_train):,}".replace(",", "."))
    log(f"Outer fold {outer_fold} probability calibration size: {len(y_proba_calib):,}".replace(",", "."))
    log(f"Outer fold {outer_fold} threshold tuning size: {len(y_threshold):,}".replace(",", "."))

    preprocessor_outer = build_preprocessor_from_schema(num_feats, cat_feats, cat_categories)
    preprocessor_outer.fit(X_final_train_raw)
    feature_names_outer_final = preprocessor_outer.get_feature_names_out().tolist()

    if feature_names_outer_final != feature_names_outer:
        raise RuntimeError(f"Feature schema mismatch in outer fold {outer_fold}.")

    X_final_train = preprocessor_outer.transform(X_final_train_raw).astype(np.float32)
    X_proba_calib = preprocessor_outer.transform(X_proba_calib_raw).astype(np.float32)
    X_threshold = preprocessor_outer.transform(X_threshold_raw).astype(np.float32)
    X_val = preprocessor_outer.transform(X_val_raw).astype(np.float32)

    X_final_train_iht, y_final_train_iht, _, outer_iht_time_min = apply_iht(
        X_final_train,
        y_final_train,
        SEED + outer_fold,
    )
    X_final_train_iht_sel = X_final_train_iht[:, consolidated_idx]
    X_proba_calib_sel = X_proba_calib[:, consolidated_idx]
    X_threshold_sel = X_threshold[:, consolidated_idx]
    X_val_sel = X_val[:, consolidated_idx]

    outer_model = fit_logreg(
        C=best_params_outer["C"],
        fit_intercept=best_params_outer["fit_intercept"],
        seed=SEED + outer_fold,
    )
    outer_model.fit(X_final_train_iht_sel, y_final_train_iht)

    y_proba_calib_raw = outer_model.predict_proba(X_proba_calib_sel)[:, 1]
    y_threshold_proba_raw = outer_model.predict_proba(X_threshold_sel)[:, 1]
    y_val_proba_raw = outer_model.predict_proba(X_val_sel)[:, 1]

    calibrator = None
    calibration_method = "none_raw_logreg_proba"
    if USE_SIGMOID_PROBA_CALIBRATION and len(y_proba_calib) > 0:
        calibrator = fit_sigmoid_calibrator(
            y_calib=y_proba_calib,
            proba_calib=y_proba_calib_raw,
            seed=SEED + outer_fold * 7000,
        )
        if calibrator is not None:
            calibration_method = "sigmoid_platt_on_outer_calibration"

    y_threshold_proba = apply_sigmoid_calibrator(calibrator, y_threshold_proba_raw)
    y_val_proba = apply_sigmoid_calibrator(calibrator, y_val_proba_raw)

    best_threshold_row, df_thresholds = optimize_threshold_constrained(
        y_true=y_threshold,
        y_proba=y_threshold_proba,
        objective=THRESHOLD_OBJECTIVE,
    )
    selected_threshold = float(best_threshold_row["Threshold"])

    df_thresholds.insert(0, "Outer_Fold", outer_fold)
    outer_threshold_rows.append(df_thresholds)
    threshold_path = os.path.join(OUTPUT_DIR, f"threshold_search_outerfold_{outer_fold}_{RUN_TAG}.csv")
    df_thresholds.to_csv(threshold_path, index=False)

    y_val_pred = (y_val_proba >= selected_threshold).astype(np.uint8)
    fold_metrics = calculate_metrics(y_val, y_val_pred, y_val_proba)

    confusion_row = {
        "Outer_Fold": outer_fold,
        "TN": fold_metrics["TN"],
        "FP": fold_metrics["FP"],
        "FN": fold_metrics["FN"],
        "TP": fold_metrics["TP"],
        "Real_Negative": fold_metrics["Real_Negative"],
        "Real_Positive": fold_metrics["Real_Positive"],
        "Predicted_Negative": fold_metrics["Predicted_Negative"],
        "Predicted_Positive": fold_metrics["Predicted_Positive"],
        "Predicted_Negative_Rate": fold_metrics["Predicted_Negative_Rate"],
        "Predicted_Positive_Rate": fold_metrics["Predicted_Positive_Rate"],
        "Threshold": selected_threshold,
        "Threshold_Selection_Mode": best_threshold_row["Threshold_Selection_Mode"],
    }
    confusion_rows.append(confusion_row)

    outer_results.append({
        "Outer_Fold": outer_fold,
        "Threshold": selected_threshold,
        "Selected_Threshold": selected_threshold,
        "Inner_Median_Threshold": consolidated_threshold,
        "Threshold_Objective": THRESHOLD_OBJECTIVE,
        "Threshold_Selection_Mode": best_threshold_row["Threshold_Selection_Mode"],
        "Threshold_Valid_Candidates": int(best_threshold_row["Threshold_Valid_Candidates"]),
        "Threshold_Total_Candidates": int(best_threshold_row["Threshold_Total_Candidates"]),
        "ThresholdTune_Precision": float(best_threshold_row["Precision"]),
        "ThresholdTune_Recall": float(best_threshold_row["Recall"]),
        "ThresholdTune_Specificity": float(best_threshold_row["Specificity"]),
        "ThresholdTune_NPV": float(best_threshold_row["NPV"]) if not pd.isna(best_threshold_row["NPV"]) else np.nan,
        "ThresholdTune_F1": float(best_threshold_row["F1"]),
        "ThresholdTune_Youden_J": float(best_threshold_row["Youden_J"]),
        "ThresholdTune_Balanced_Accuracy": float(best_threshold_row["Balanced_Accuracy"]),
        "ThresholdTune_Predicted_Positive_Rate": float(best_threshold_row["Predicted_Positive_Rate"]),
        "ThresholdTune_Predicted_Negative_Rate": float(best_threshold_row["Predicted_Negative_Rate"]),
        "Calibration_Method": calibration_method,
        "Probability_Calibrated": bool(calibrator is not None),
        "N_Selected_Features": TOP_K_FEATURES,
        "Inner_Best_C": best_params_outer["C"],
        "Inner_Best_fit_intercept": best_params_outer["fit_intercept"],
        "Final_Train_Size": int(len(y_final_train)),
        "Proba_Calibration_Size": int(len(y_proba_calib)),
        "Threshold_Tuning_Size": int(len(y_threshold)),
        "Inner_IHT_Time_Min_Mean": float(np.mean(inner_iht_times)),
        "Outer_IHT_Time_Min": outer_iht_time_min,
        **fold_metrics,
    })

    log(
        f"OUTER FOLD {outer_fold} CONFUSION MATRIX [[TN, FP], [FN, TP]] = "
        f"[[{fold_metrics['TN']}, {fold_metrics['FP']}], [{fold_metrics['FN']}, {fold_metrics['TP']}]]"
    )
    log(
        f"OUTER FOLD {outer_fold} COMPLETE | "
        f"Precision={fold_metrics['Precision']:.6f} | "
        f"Recall={fold_metrics['Recall']:.6f} | "
        f"Specificity={fold_metrics['Specificity']:.6f} | "
        f"NPV={fold_metrics['NPV'] if not pd.isna(fold_metrics['NPV']) else np.nan:.6f} | "
        f"F1={fold_metrics['F1']:.6f} | "
        f"ROC_AUC={fold_metrics['ROC_AUC']:.6f} | "
        f"Threshold={selected_threshold:.6f} | "
        f"PredPosRate={fold_metrics['Predicted_Positive_Rate']:.4f} | "
        f"PredNegRate={fold_metrics['Predicted_Negative_Rate']:.4f} | "
        f"ThresholdMode={best_threshold_row['Threshold_Selection_Mode']} | "
        f"Time={(time.time() - outer_start)/60.0:.1f}m"
    )

    if np.isfinite(fold_metrics["ROC_AUC"]) and fold_metrics["ROC_AUC"] > best_outer_fold_auc:
        best_outer_fold_auc = fold_metrics["ROC_AUC"]
        best_outer_artifacts = {
            "outer_fold": outer_fold,
            "model": outer_model,
            "preprocessor": preprocessor_outer,
            "calibrator": calibrator,
            "calibration_method": calibration_method,
            "selected_threshold": selected_threshold,
            "threshold_objective": THRESHOLD_OBJECTIVE,
            "threshold_row": best_threshold_row,
            "X_val_sel": X_val_sel,
            "y_val": y_val.copy(),
            "selected_feature_names": consolidated_features.copy(),
            "selected_feature_indices": consolidated_idx.copy(),
            "best_params": best_params_outer.copy(),
        }

    outer_artifact_path = os.path.join(OUTPUT_DIR, f"artifact_outerfold_{outer_fold}_{RUN_TAG}.pkl")
    joblib.dump({
        "outer_fold": outer_fold,
        "model": outer_model,
        "preprocessor": preprocessor_outer,
        "calibrator": calibrator,
        "calibration_method": calibration_method,
        "selected_threshold": selected_threshold,
        "threshold_objective": THRESHOLD_OBJECTIVE,
        "threshold_row": best_threshold_row,
        "selected_feature_names": consolidated_features.copy(),
        "selected_feature_indices": consolidated_idx.copy(),
        "best_params": best_params_outer.copy(),
    }, outer_artifact_path)

    # save per-outer fold consolidated feature summary
    outer_feat_path = os.path.join(OUTPUT_DIR, f"inner_feature_summary_outerfold_{outer_fold}_{RUN_TAG}.csv")
    inner_feature_summary.to_csv(outer_feat_path, index=False)

    del X_cal_raw, X_val_raw
    del X_final_train_raw, y_final_train, X_proba_calib_raw, y_proba_calib, X_threshold_raw, y_threshold
    del X_final_train, X_proba_calib, X_threshold, X_val
    del X_final_train_iht, y_final_train_iht, X_final_train_iht_sel
    del X_proba_calib_sel, X_threshold_sel, X_val_sel, outer_model, calibrator
    del y_proba_calib_raw, y_threshold_proba_raw, y_val_proba_raw, y_threshold_proba, y_val_proba, y_val_pred, df_thresholds
    gc.collect()



##########################################################################################
STARTING OUTER FOLD 1/10
##########################################################################################

OUTER FOLD 1 - CALIBRATION
--------------------------
Total: 800.000
Class 0: 682.532 (85.3165%)
Class 1: 117.468 (14.6835%)
Imbalance ratio (maj/min): 5.8104:1

OUTER FOLD 1 - VALIDATION
-------------------------
Total: 711.380
Class 0: 606.925 (85.3166%)
Class 1: 104.455 (14.6834%)
Imbalance ratio (maj/min): 5.8104:1
Outer fold 1 schema | numeric=15 | categorical=10

OUTER 1 | INNER 1/10
IHT distribution | before: class0=213,291, class1=36,709 | after: class0=73,418, class1=36,709 | maj/min=2.0000:1
Inner fold IHT time: 0.05m

OUTER 1 | INNER 2/10
IHT distribution | before: class0=213,291, class1=36,709 | after: class0=73,418, class1=36,709 | maj/min=2.0000:1
Inner fold IHT time: 0.05m

OUTER 1 | INNER 3/10
IHT distribution | before: class0=213,291, class1=36,709 | after: class0

In [39]:

# ------------------ Save outputs ------------------
df_outer = pd.DataFrame(outer_results)
results_path = os.path.join(OUTPUT_DIR, f"outer_fold_results_{RUN_TAG}.csv")
df_outer.to_csv(results_path, index=False)

summary_df = summarize_outer_results(df_outer)
summary_path = os.path.join(OUTPUT_DIR, f"summary_mean_std_{RUN_TAG}.csv")
summary_df.to_csv(summary_path, index=False)

confusion_df = pd.DataFrame(confusion_rows)
confusion_path = os.path.join(OUTPUT_DIR, f"confusion_matrices_{RUN_TAG}.csv")
confusion_df.to_csv(confusion_path, index=False)

pooled_summary_df = summarize_pooled_confusion(df_outer)
pooled_summary_path = os.path.join(OUTPUT_DIR, f"pooled_confusion_summary_{RUN_TAG}.csv")
pooled_summary_df.to_csv(pooled_summary_path, index=False)

if outer_threshold_rows:
    threshold_all_df = pd.concat(outer_threshold_rows, ignore_index=True)
else:
    threshold_all_df = pd.DataFrame()
threshold_all_path = os.path.join(OUTPUT_DIR, f"threshold_search_all_outerfolds_{RUN_TAG}.csv")
threshold_all_df.to_csv(threshold_all_path, index=False)

outer_feature_df = pd.DataFrame(outer_feature_rows)
outer_feature_path = os.path.join(OUTPUT_DIR, f"outer_consolidated_top{TOP_K_FEATURES}_{RUN_TAG}.csv")
outer_feature_df.to_csv(outer_feature_path, index=False)

inner_feature_df = pd.DataFrame(outer_inner_feature_rows)
inner_feature_path = os.path.join(OUTPUT_DIR, f"inner_selected_features_{RUN_TAG}.csv")
inner_feature_df.to_csv(inner_feature_path, index=False)


In [40]:

# candidate diagnostics
candidate_diag_df = pd.concat(outer_inner_candidate_rows, ignore_index=True)
candidate_diag_path = os.path.join(OUTPUT_DIR, f"candidate_diagnostics_{RUN_TAG}.csv")
candidate_diag_df.to_csv(candidate_diag_path, index=False)


In [41]:

# global final consolidated top20 across outer folds
global_feature_summary = (
    outer_feature_df.groupby("Feature")
    .agg(
        Outer_Selection_Frequency=("Feature", "count"),
        Mean_Outer_Rank=("Rank_Within_Outer", "mean"),
        Mean_Outer_Importance=("Mean_Inner_Importance", "mean"),
        Std_Outer_Importance=("Mean_Inner_Importance", "std"),
    )
    .reset_index()
)
global_feature_summary = global_feature_summary.sort_values(
    by=["Outer_Selection_Frequency", "Mean_Outer_Importance", "Mean_Outer_Rank"],
    ascending=[False, False, True],
).reset_index(drop=True)

global_feature_summary_path = os.path.join(OUTPUT_DIR, f"global_feature_summary_{RUN_TAG}.csv")
global_feature_summary.to_csv(global_feature_summary_path, index=False)

global_top20_df = global_feature_summary.head(TOP_K_FEATURES).copy()
global_top20_path = os.path.join(OUTPUT_DIR, f"global_top{TOP_K_FEATURES}_{RUN_TAG}.csv")
global_top20_df.to_csv(global_top20_path, index=False)


In [42]:

# save model from best outer fold if requested
if best_outer_artifacts is not None:
    best_model_path = os.path.join(OUTPUT_DIR, f"best_outer_model_{RUN_TAG}.pkl")
    best_artifacts_path = os.path.join(OUTPUT_DIR, f"best_outer_artifacts_{RUN_TAG}.pkl")
    joblib.dump(best_outer_artifacts["model"], best_model_path)
    joblib.dump(best_outer_artifacts, best_artifacts_path)

    best_meta = {
        "best_outer_fold": int(best_outer_artifacts["outer_fold"]),
        "best_outer_fold_auc": float(best_outer_fold_auc),
        "best_outer_selected_features": best_outer_artifacts["selected_feature_names"],
        "selected_threshold": float(best_outer_artifacts["selected_threshold"]),
        "threshold_objective": best_outer_artifacts["threshold_objective"],
        "calibration_method": best_outer_artifacts["calibration_method"],
        "best_params": best_outer_artifacts["best_params"],
    }
    with open(os.path.join(OUTPUT_DIR, f"best_outer_model_metadata_{RUN_TAG}.json"), "w", encoding="utf-8") as f:
        json.dump(best_meta, f, ensure_ascii=False, indent=2, default=str)

    if RUN_OUTER_BEST_FOLD_PERMUTATION:
        X_perm, y_perm = make_perm_subset(
            best_outer_artifacts["X_val_sel"],
            best_outer_artifacts["y_val"],
            OUTER_PERM_MAX_SAMPLES,
            SEED,
        )
        perm_results = permutation_importance(
            estimator=best_outer_artifacts["model"],
            X=X_perm,
            y=y_perm,
            scoring=roc_auc_proba_scorer,
            n_repeats=OUTER_PERM_REPEATS,
            random_state=SEED,
            n_jobs=1,
            max_samples=1.0,
        )
        df_perm_best_outer = pd.DataFrame({
            "Feature": best_outer_artifacts["selected_feature_names"],
            "Importance_Mean": perm_results.importances_mean,
            "Importance_Std": perm_results.importances_std,
        }).sort_values(by="Importance_Mean", ascending=False)
        df_perm_best_outer.to_csv(
            os.path.join(OUTPUT_DIR, f"best_outer_permutation_{RUN_TAG}.csv"),
            index=False,
        )


In [43]:

config = {
    "RUN_TAG": RUN_TAG,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "TARGET_COL": TARGET_COL,
    "N_OUTER_FOLDS": N_OUTER_FOLDS,
    "N_INNER_FOLDS": N_INNER_FOLDS,
    "TOP_K_FEATURES": TOP_K_FEATURES,
    "IHT_SAMPLING_STRATEGY": IHT_SAMPLING_STRATEGY,
    "IHT_CV": IHT_CV,
    "OUTER_CALIBRATION_CAP": OUTER_CALIBRATION_CAP,
    "INNER_TRAIN_CAP": INNER_TRAIN_CAP,
    "NUMERIC_SCALER": "MinMaxScaler",
    "SEARCH_SIZE_CAP": SEARCH_SIZE_CAP,
    "C_CANDIDATES": C_CANDIDATES,
    "FIT_INTERCEPT_CANDIDATES": FIT_INTERCEPT_CANDIDATES,
    "PERM_RANK_MAX_SAMPLES": PERM_RANK_MAX_SAMPLES,
    "PERM_RANK_REPEATS": PERM_RANK_REPEATS,
    "THRESHOLD_OBJECTIVE": THRESHOLD_OBJECTIVE,
    "MANUAL_THRESHOLDS": MANUAL_THRESHOLDS,
    "MIN_THRESHOLD": MIN_THRESHOLD,
    "MIN_SPECIFICITY_FOR_THRESHOLD": MIN_SPECIFICITY_FOR_THRESHOLD,
    "MIN_PRECISION_MARGIN_OVER_PREVALENCE": MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    "MAX_PREDICTED_POSITIVE_RATE": MAX_PREDICTED_POSITIVE_RATE,
    "MIN_PREDICTED_NEGATIVE_RATE": MIN_PREDICTED_NEGATIVE_RATE,
    "USE_SIGMOID_PROBA_CALIBRATION": USE_SIGMOID_PROBA_CALIBRATION,
    "FINAL_PROBA_CALIBRATION_SIZE": FINAL_PROBA_CALIBRATION_SIZE,
    "FINAL_THRESHOLD_SIZE": FINAL_THRESHOLD_SIZE,
}
with open(os.path.join(OUTPUT_DIR, f"config_{RUN_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

log("\n" + "#" * 90)
log("NESTED-CV WORKFLOW COMPLETE")
log("#" * 90)
log(f"Outer-fold results saved: {results_path}")
log(f"Mean/std summary saved: {summary_path}")
log(f"Outer consolidated top-{TOP_K_FEATURES} saved: {outer_feature_path}")
log(f"Global feature summary saved: {global_feature_summary_path}")
log(f"Global top-{TOP_K_FEATURES} saved: {global_top20_path}")
log(f"Candidate diagnostics saved: {candidate_diag_path}")



##########################################################################################
NESTED-CV WORKFLOW COMPLETE
##########################################################################################
Outer-fold results saved: ../data/processed/outer_fold_results_logreg_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Mean/std summary saved: ../data/processed/summary_mean_std_logreg_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Outer consolidated top-20 saved: ../data/processed/outer_consolidated_top20_logreg_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Global feature summary saved: ../data/processed/global_feature_summary_logreg_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Global top-20 saved: ../data/processed/global_top20_logreg_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Candidate diagnostics saved: ../data/processed/candidate_diagnostics_logreg_iht_refreshe

In [44]:

log("\nOUTER FOLD METRICS")
log(str(df_outer[["Outer_Fold", "Precision", "Recall", "F1", "ROC_AUC", "Threshold"]]))
log("\nMEAN / STD SUMMARY")
log(str(summary_df))
log("\nGLOBAL TOP FEATURES")
log(str(global_top20_df))


OUTER FOLD METRICS
   Outer_Fold  Precision    Recall        F1   ROC_AUC  Threshold
0           1   0.231749  0.617778  0.337057  0.679910   0.148267
1           2   0.232758  0.618640  0.338251  0.681680   0.148015
2           3   0.228005  0.624010  0.333979  0.677159   0.144323
3           4   0.221588  0.672567  0.333348  0.682101   0.138371
4           5   0.223383  0.657881  0.333520  0.680150   0.140013
5           6   0.231017  0.625708  0.337446  0.679864   0.145072
6           7   0.230174  0.611176  0.334407  0.678482   0.150024
7           8   0.228399  0.653385  0.338479  0.682058   0.141122
8           9   0.222917  0.658322  0.333056  0.679559   0.140000
9          10   0.230742  0.625896  0.337180  0.680007   0.145097

MEAN / STD SUMMARY
                     Metric       Mean        Std        Min         Max  \
0                  Accuracy   0.629973   0.014650   0.605003    0.644574   
1                 Precision   0.228073   0.004033   0.221588    0.232758   
2     